# 11 — Medical Language Model Pretraining

This notebook trains a **fresh GPT model from scratch** on the combined medical corpus
(clinical guidelines + PubMedQA abstracts) built in notebook 10.

**Pipeline:**
1. Initialise a fresh GPT model with randomly initialised weights
2. Load the tokenized medical splits (memory-mapped)
3. Pretrain for 20,000 steps with warmup → cosine LR schedule
4. Plot the loss curve and generate sample medical completions

**Expected outcome:** A model that generates fluent medical prose and has internalised
domain vocabulary — ready to be fine-tuned on Q&A tasks in notebook 12.

## 1 — Setup

In [ ]:
import os, sys, math

# ── Detect environment ────────────────────────────────────────────────────
try:
    import google.colab
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/slm-learning"
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

# ── Clone repo and install deps (Colab only, first run) ──────────────────
if ON_COLAB and not os.path.exists(REPO_DIR):
    import subprocess
    subprocess.run([
        "pip", "install", "-q",
        "tokenizers", "datasets", "tqdm", "matplotlib", "huggingface_hub"
    ], check=True)
    token = ""   # paste your GitHub PAT here if repo is private
    url   = f"https://{token}@github.com/Aman2394/slm-from-scratch.git" if token \
            else "https://github.com/Aman2394/slm-from-scratch.git"
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

# ── Add repo root to Python path ─────────────────────────────────────────
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
import config as cfg
cfg.make_dirs()
cfg.print_config()

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"\nTraining on: {device}")
if device == "cuda":
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2 — Why Train from Scratch?

The TinyStories model was trained on children's stories — a completely different distribution
from medical textbooks. Warm-starting from it would bring vocabulary assumptions and weight
distributions tuned for a very different domain.

Training from scratch on medical text gives the model:
- **Clean slate embeddings** matched to the medical tokenizer vocabulary
- **Domain-aligned weight distributions** from the first gradient step
- **No interference** from story-style language patterns

This is the same approach used by dedicated medical LMs like
[BioMedLM](https://arxiv.org/abs/2305.09530) (Stanford) and
[Meditron](https://arxiv.org/abs/2311.16079) (EPFL) — purpose-built rather than adapted.

**Initial loss sanity check:**  
With randomly initialised weights, the model assigns roughly uniform probability across all
8,192 tokens. The expected initial cross-entropy loss is `log(8192) ≈ 9.01`.

## 3 — Initialise a Fresh GPT Model

We create a `GPTConfig`, override only `block_size` to 512 (medical documents are longer
than TinyStories), and initialise a new `GPT` from random weights.

> **VRAM note:** `block_size=512` uses ~4× the attention memory of the default 256
> (attention scales quadratically with sequence length).  
> If you hit OOM, reduce `MED_BLOCK_SIZE` to `384` or set `cfg.MED_DAPT_GRAD_ACCUM_STEPS = 2`.

In [ ]:
from model.gpt import GPT, GPTConfig
import config as cfg

# ── Block size override for medical text ─────────────────────────────────────
# Medical guidelines and abstracts are typically 300–600 tokens.
# block_size=512 lets the model attend over a full document in one pass.
# WARNING: 4× attention memory vs default 256 — may require grad_accum_steps=2 on T4.
MED_BLOCK_SIZE = 512

# GPTConfig reads defaults from config.py; we only override block_size here.
# All other hyperparams (n_layer, n_head, n_embd, dropout) stay at config defaults.
med_config = GPTConfig()
med_config.block_size = MED_BLOCK_SIZE

# Fresh random initialisation — no weights transferred from any prior model
model = GPT(med_config).to(device)

print(f"Model parameters : {model.num_parameters() / 1e6:.2f}M")
print(f"block_size       : {med_config.block_size}")
print(f"vocab_size       : {med_config.vocab_size}")
print(f"n_layer          : {med_config.n_layer}")
print(f"n_head           : {med_config.n_head}")
print(f"n_embd           : {med_config.n_embd}")

# Sanity: expected initial loss for a random model = log(vocab_size)
expected_initial_loss = math.log(med_config.vocab_size)
print(f"\nExpected initial loss (random weights): {expected_initial_loss:.3f}")

In [ ]:
# ── Quick forward-pass sanity check ──────────────────────────────────────────
# Verify the model initialised correctly before committing to a long training run.
x_test = torch.randint(0, med_config.vocab_size, (2, MED_BLOCK_SIZE)).to(device)
with torch.no_grad():
    logits = model(x_test)
print("Forward pass OK")
print(f"  Input shape  : {x_test.shape}")
print(f"  Output shape : {logits.shape}   # expected (2, {MED_BLOCK_SIZE}, {med_config.vocab_size})")

## 4 — Data Loaders

We load the medical train and val token arrays as memory-mapped numpy arrays.
`np.memmap` opens the file without loading it into RAM; the OS pages in slices on demand.
This lets us train on a corpus larger than RAM — critical on Colab with limited memory.

In [ ]:
import numpy as np
import config as cfg

train_tokens = np.memmap(cfg.MED_TRAIN_TOKENS, dtype=np.uint16, mode="r")
val_tokens   = np.memmap(cfg.MED_VAL_TOKENS,   dtype=np.uint16, mode="r")

print(f"Train tokens : {len(train_tokens):>12,}  ({len(train_tokens)/1e6:.1f}M)")
print(f"Val tokens   : {len(val_tokens):>12,}  ({len(val_tokens)/1e6:.1f}M)")


def get_batch(data, block_size, batch_size):
    """
    Sample a random batch of (input, target) pairs from a flat token array.

    Each row is `block_size` consecutive tokens; the target is the same
    sequence shifted right by 1 — the standard causal LM objective.

    Args:
        data       : np.memmap uint16 token array
        block_size : context length (tokens per sequence)
        batch_size : number of sequences per batch

    Returns:
        x, y: torch.Tensor of shape (batch_size, block_size), int64, on device
    """
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x  = torch.stack([torch.from_numpy(data[i     : i + block_size    ].astype(np.int64)) for i in ix])
    y  = torch.stack([torch.from_numpy(data[i + 1 : i + block_size + 1].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)


# Test batch
xb, yb = get_batch(train_tokens, MED_BLOCK_SIZE, cfg.MED_DAPT_BATCH_SIZE)
print(f"\nSample batch — x: {xb.shape}  y: {yb.shape}  dtype: {xb.dtype}")

## 5 — LR Schedule & Optimizer

We use **linear warmup → cosine decay**, the standard schedule for transformer pretraining
(GPT-2, GPT-3, Llama all use this).

- **Warmup (300 steps):** prevents large gradient updates in the first steps when all
  embeddings are random and gradients are noisy
- **Cosine decay:** smoothly reduces LR to 10% of peak at the end of training,
  allowing fine-grained weight adjustment without oscillation

All hyperparameters are sourced from `config.py`.

In [ ]:
import math
import torch
import config as cfg


def get_lr(step):
    """
    Compute learning rate for a given training step.

    Schedule: linear warmup for cfg.MED_DAPT_WARMUP_STEPS steps,
    then cosine decay to 10% of peak LR at cfg.MED_DAPT_MAX_STEPS.

    Args:
        step: current training step (0-indexed)

    Returns:
        float: learning rate at this step
    """
    warmup = cfg.MED_DAPT_WARMUP_STEPS
    total  = cfg.MED_DAPT_MAX_STEPS
    peak   = cfg.MED_DAPT_LR

    if step < warmup:
        # Linear ramp: avoids explosive gradients when embeddings are freshly random
        return peak * (step + 1) / warmup

    # Cosine decay from peak to 10% of peak
    progress = (step - warmup) / max(1, total - warmup)
    return peak * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))


# AdamW: weight decay regularises all parameters except biases and LayerNorm scales
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.MED_DAPT_LR,
    weight_decay=cfg.MED_DAPT_WEIGHT_DECAY,
    betas=(0.9, 0.95),   # standard GPT-3 betas
)

# Spot-check: LR at key steps
for s in [0, cfg.MED_DAPT_WARMUP_STEPS, cfg.MED_DAPT_MAX_STEPS // 2, cfg.MED_DAPT_MAX_STEPS - 1]:
    print(f"  step {s:>6}: lr = {get_lr(s):.2e}")

## 6 — Training Loop

**Training details:**
- 20,000 optimizer steps with batch_size=16 and block_size=512
- **Gradient accumulation** over `cfg.MED_DAPT_GRAD_ACCUM_STEPS` micro-batches per optimizer step.
  Effective batch = `batch_size × grad_accum_steps`. Set `grad_accum_steps=2` if OOM on T4.
- Evaluate on `cfg.MED_DAPT_EVAL_BATCHES` val batches every `cfg.MED_DAPT_EVAL_INTERVAL` steps
- Checkpoint every `cfg.MED_DAPT_SAVE_INTERVAL` steps — auto-resumes if Colab disconnects
- Gradient clipping at `cfg.MED_DAPT_GRAD_CLIP` prevents explosions during embedding warmup

**Expected loss trajectory:**
| Steps | Loss | Notes |
|-------|------|-------|
| 0 | ~9.0 | Random weights — matches `log(8192)` |
| 300 | ~6–7 | Warmup complete, embeddings forming |
| 5k | ~4.5 | Domain syntax learned |
| 20k | ~3.5–4.0 | Good domain model for 25M params |

In [ ]:
import torch.nn.functional as F


@torch.no_grad()
def estimate_val_loss():
    """
    Estimate validation loss by averaging over cfg.MED_DAPT_EVAL_BATCHES random batches.

    Multiple batches reduce variance in the estimate. Dropout is disabled
    during eval by temporarily setting the model to eval mode.

    Returns:
        float: mean cross-entropy loss on the validation set
    """
    model.eval()
    losses = []
    for _ in range(cfg.MED_DAPT_EVAL_BATCHES):
        xv, yv = get_batch(val_tokens, MED_BLOCK_SIZE, cfg.MED_DAPT_BATCH_SIZE)
        logits  = model(xv)
        loss    = F.cross_entropy(logits.view(-1, med_config.vocab_size), yv.view(-1))
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

In [ ]:
# ── Resume from checkpoint if one exists ──────────────────────────────────────
# Colab sessions disconnect without warning. This block means we never lose
# progress — just re-run the cell after reconnecting.
start_step = 0
if os.path.exists(cfg.MED_DAPT_CKPT):
    print(f"Resuming from checkpoint: {cfg.MED_DAPT_CKPT}")
    ckpt = torch.load(cfg.MED_DAPT_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_step = ckpt["step"]
    print(f"  → resuming from step {start_step:,}")
else:
    print("No checkpoint found — starting from step 0.")

print(f"Will train steps {start_step:,} → {cfg.MED_DAPT_MAX_STEPS:,}")

In [ ]:
%%time
# ── Medical Pretraining Loop ──────────────────────────────────────────────────
import torch.nn.functional as F

grad_accum = cfg.MED_DAPT_GRAD_ACCUM_STEPS   # micro-batches per optimizer step
# Effective batch size = MED_DAPT_BATCH_SIZE × grad_accum_steps
# With grad_accum=1 (default): effective batch = 16
# With grad_accum=2 (OOM fallback): effective batch = 32, half the VRAM per step
print(f"Gradient accumulation steps : {grad_accum}")
print(f"Effective batch size         : {cfg.MED_DAPT_BATCH_SIZE * grad_accum}")

model.train()
loss_log = []   # (step, train_loss, val_loss) tuples for the loss curve plot

for step in range(start_step, cfg.MED_DAPT_MAX_STEPS):

    # ── Update LR according to warmup-cosine schedule ────────────────────────
    lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    # ── Gradient accumulation inner loop ─────────────────────────────────────
    # We zero gradients once per optimizer step, then accumulate over
    # grad_accum micro-batches before updating weights.
    # Dividing loss by grad_accum ensures the gradient magnitude is equivalent
    # to computing the loss over one large batch of size batch_size × grad_accum.
    optimizer.zero_grad()
    accum_loss = 0.0

    for micro_step in range(grad_accum):
        xb, yb = get_batch(train_tokens, MED_BLOCK_SIZE, cfg.MED_DAPT_BATCH_SIZE)
        logits  = model(xb)
        # Scale loss so gradients are averaged (not summed) across micro-batches
        loss    = F.cross_entropy(logits.view(-1, med_config.vocab_size), yb.view(-1)) / grad_accum
        loss.backward()
        accum_loss += loss.item()   # already scaled, so sum = mean over micro-batches

    # ── Gradient clipping ────────────────────────────────────────────────────
    # Clip after accumulation so the norm is computed over the full accumulated
    # gradient, not each micro-batch separately.
    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MED_DAPT_GRAD_CLIP)

    optimizer.step()

    # ── Periodic logging ─────────────────────────────────────────────────────
    if step % 100 == 0:
        print(f"step {step:>6} | lr {lr:.2e} | train loss {accum_loss:.4f}")

    # ── Validation ───────────────────────────────────────────────────────────
    if step % cfg.MED_DAPT_EVAL_INTERVAL == 0 and step > 0:
        val_loss = estimate_val_loss()
        val_ppl  = math.exp(val_loss)
        loss_log.append((step, accum_loss, val_loss))
        print(f"step {step:>6} | VAL loss {val_loss:.4f} | val ppl {val_ppl:.1f}")

    # ── Checkpoint ───────────────────────────────────────────────────────────
    if step % cfg.MED_DAPT_SAVE_INTERVAL == 0 and step > 0:
        torch.save(
            {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": step},
            cfg.MED_DAPT_CKPT,
        )
        print(f"  Checkpoint saved → {cfg.MED_DAPT_CKPT}")

# ── Save final weights ────────────────────────────────────────────────────────
torch.save(model.state_dict(), cfg.MED_DAPT_FINAL_CKPT)
print(f"\nPretraining complete. Final model → {cfg.MED_DAPT_FINAL_CKPT}")

## 7 — Loss Curve

Plot validation loss at each eval checkpoint.

Expected trajectory:
- **Steps 0–300 (warmup):** loss drops rapidly as embeddings learn basic token co-occurrences
- **Steps 300–5k:** loss falls from ~6–7 to ~4.5 as the model adapts to medical syntax
- **Steps 5k–20k:** slow, steady improvement toward ~3.5–4.0

In [ ]:
if not loss_log:
    print("No loss log — re-run the training loop cell to populate it.")
else:
    try:
        import matplotlib.pyplot as plt
        steps      = [s  for s, _, _ in loss_log]
        train_loss = [tl for _, tl, _ in loss_log]
        val_loss   = [vl for _, _, vl in loss_log]

        plt.figure(figsize=(10, 4))
        plt.plot(steps, train_loss, label="train", linewidth=1.2, color="steelblue")
        plt.plot(steps, val_loss,   label="val",   linewidth=1.2, color="coral",     linestyle="--")
        plt.xlabel("Training Step")
        plt.ylabel("Cross-Entropy Loss")
        plt.title("Medical Pretraining — Loss Curve")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    except ImportError:
        print(f"{'Step':>8}  {'Train Loss':>12}  {'Val Loss':>10}")
        print("-" * 40)
        for step, tl, vl in loss_log:
            print(f"{step:>8}  {tl:>12.4f}  {vl:>10.4f}")

## 8 — Sample Completions

Test the pretrained model with 5 medical prompts. At this stage the model should produce
domain-appropriate vocabulary and syntactically plausible sentences, but will not yet
follow an instruction format — that comes in notebook 12 (SFT).

In [ ]:
import textwrap
from tokenizer.preprocess import load_tokenizer
from generation.sampler import generate, encode_prompt
import config as cfg

med_tok = load_tokenizer(cfg.MED_TOKENIZER_VOCAB, cfg.MED_TOKENIZER_MERGES)
model.eval()

MEDICAL_PROMPTS = [
    "Myocardial infarction occurs when",
    "The mechanism of action of beta-blockers is",
    "Pneumonia is diagnosed by",
    "Insulin resistance is characterized by",
    "The first-line treatment for hypertension is",
]

for prompt in MEDICAL_PROMPTS:
    x   = encode_prompt(prompt, med_tok, device)
    out = generate(
        model, x, med_tok,
        block_size=MED_BLOCK_SIZE,
        max_new_tokens=120,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
        repetition_penalty=1.1,
    )
    text = med_tok.decode(out[0].tolist())
    print(f"\nPROMPT: {prompt}")
    print(textwrap.fill(text, 80))
    print("─" * 80)

model.train()

## Next Steps

The pretrained medical model is saved at `cfg.MED_DAPT_FINAL_CKPT`. It generates fluent
medical prose but does not yet follow an instruction format. Next:

- **Notebook 12** — Supervised Fine-Tuning (SFT) on MedQA + PubMedQA instruction pairs
- **Notebook 13** — Evaluation: MCQ accuracy and medical perplexity benchmarks

To push the pretrained weights to HuggingFace Hub (optional, before SFT):
```python
from huggingface_hub import HfApi
HfApi().upload_file(
    path_or_fileobj=cfg.MED_DAPT_FINAL_CKPT,
    path_in_repo="med_pretrain_final.pt",
    repo_id=cfg.MED_HF_MODEL_REPO,
    repo_type="model",
)
```